# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one pseudonymized content item (one snapshot per item in the starter CSV). The starter CSV contains trailing-90-day summary fields (see `docs/data-dictionary.md`) and represents each content item's recent 90-day window; the warehouse tables use different grains — check windows before joining.

In [ ]:
# Load starter CSV and verify grain and window info
import pandas as pd
from pathlib import Path
data_path = Path('data/raw/content_refresh_anonymized.csv')
if not data_path.exists():
	print(f"Missing file: {data_path}. Run from repo root.")
else:
	df = pd.read_csv(data_path)
	print('rows,cols =', df.shape)
	# check for obvious date columns
	date_cols = [c for c in df.columns if 'date' in c.lower() or 'report' in c.lower()]
	print('Date-like columns found:', date_cols)
	if date_cols:
		for c in date_cols:
			try:
				s = pd.to_datetime(df[c], errors='coerce')
				print(c, 'min/max ->', s.min(), s.max(), 'non-null:', s.notna().sum())
			except Exception:
				print('Could not parse', c)
	# grain check: content_id uniqueness
	if 'content_id' in df.columns:
		dup = df.groupby(['content_id']).size().reset_index(name='c').query('c>1')
		print('Duplicate content_id rows (should be zero for starter CSV):', len(dup))
	else:
		print('No content_id column found — check data dictionary.')

## 2. Fields: feature / label / context / excluded

Below is a suggested split for common columns. Run the verification cell to produce per-column missingness and to confirm these assignments on the real CSV.

- Feature: measurable signals available before the decision (example: `pageviews_prev90`, `ctr`, `engagement_rate`, `word_count`, `avg_position` — with gotchas).
- Label / proxy: observed outcomes computed from later windows (example: `refreshed_within_30d`, `is_declining_label` — beware derived labels).
- Context: grouping/joining keys or stable metadata (example: `content_id`, `client_id`, `content_type`, `publish_date`).
- Excluded: any product flags, privacy columns, or fields derived from `trend_pct`/`trend_direction` (these are derived and should not be used as features).

In [ ]:
# Produce a verification table: column categories, missingness, sample values
if 'df' not in globals():
	print('Run the loader cell above first to create `df`.')
else:
	cols = df.columns.tolist()
	# simple heuristics
	context = [c for c in cols if 'id' in c.lower() or 'content' in c.lower() or 'client' in c.lower()]
	label_candidates = [c for c in cols if 'refresh' in c.lower() or 'declin' in c.lower() or 'trend_direction' in c.lower()]
	rate_cols = [c for c in cols if any(x in c.lower() for x in ['ctr','engagement_rate','scroll_rate','ai_traffic_pct','trend_pct'])]
	numeric = df.select_dtypes(include=['number']).columns.tolist()

	summary = []
	for c in cols:
		missing = df[c].isna().mean()
		uniq = df[c].nunique(dropna=False)
		sample = df[c].dropna().unique()[:5].tolist()
		summary.append({'column':c,'missing_frac':missing,'n_unique':uniq,'sample_vals':sample})

	summary_df = pd.DataFrame(summary).sort_values('missing_frac', ascending=False)
	display(summary_df.head(20))

	print('\nContext columns (heuristic):', context)
	print('Label candidates (heuristic):', label_candidates)
	print('Rate-like columns (may be scaled ×100):', rate_cols)

## 3. Verify it with queries (grain, counts, missing values, windows)

Run the checks below: grain uniqueness, overall counts, missingness by `content_type` (to catch patterned missingness), and rate-column scaling checks.

In [ ]:
if 'df' not in globals():
	print('Run the loader cell first.')
else:
	# Grain check: ensure one row per content_id
	if 'content_id' in df.columns:
		dup = df.groupby('content_id').size().reset_index(name='count').query('count>1')
		print('Duplicate content_id rows:', len(dup))
	else:
		print('No content_id column to check grain.')

	# Counts
	print('\nTotal rows:', len(df))
	if 'client_id' in df.columns:
		print('Clients represented:', df['client_id'].nunique())
		print('Rows per client (sample):')
		print(df.groupby('client_id').size().sort_values(ascending=False).head())

	# Missingness by content_type
	if 'content_type' in df.columns:
		miss_by_type = df.isna().groupby(df['content_type']).mean()
		print('\nMissingness fraction by content_type (sample):')
		display(miss_by_type.head())

	# Rate columns scaling check (some rates are recorded as percent ×100)
	rate_cols = [c for c in df.columns if any(x in c.lower() for x in ['ctr','engagement_rate','scroll_rate','ai_traffic_pct','trend_pct'])]
	for c in rate_cols:
		maxi = df[c].dropna().abs().max()
		print(f'{c}: max={maxi} (if >10 likely percent-scale, e.g., 0.76==0.76%)')

	# avg_position special case
	if 'avg_position' in df.columns:
		zero_count = (df['avg_position']==0).sum()
		print('\navg_position zeros (meaning no data):', zero_count)

## 4. Data limits

What this data cannot tell you and important gotchas:

- Per-client history depth varies; you cannot assume equal history across clients.
- `avg_position = 0` indicates missing position data, not position zero.
- Rate columns (`ctr`, `engagement_rate`, `ai_traffic_pct`, `trend_pct`) are scaled; interpret accordingly.
- `trend_direction` and `trend_pct` are used to derive `is_declining_label` in the starter set — do not use these derived fields as features (label trap).
- GA4 columns may be zero-filled before a client's GA4 start date — filter on availability flags when present.

In [ ]:
# Quick automated checks for common gotchas
if 'df' not in globals():
	print('Run the loader cell first.')
else:
	# Check for label trap usage
	if 'is_declining_label' in df.columns and ('trend_pct' in df.columns or 'trend_direction' in df.columns):
		print('Warning: `is_declining_label` appears derived from trend columns — do not use trend_pct/direction as features.')

	# Check for rate columns > 1 suggesting percent scaling
	for c in df.columns:
		if any(x in c.lower() for x in ['ctr','engagement_rate','ai_traffic_pct','trend_pct']):
			s = df[c].dropna().abs()
			if not s.empty and s.max() > 1:
				print(f'Column {c} has max {s.max():.3f} — values likely represent percent-scale (×100).')

	print('\nDone automated checks.')

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.